In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, os, shutil
import pandas as pd

# === CONFIGURATION ===
base_download_dir = os.path.join(os.getcwd(), "TenderTiger_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

# Chrome setup
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "profile.default_content_settings.popups": 0,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)

# === START BROWSER ===
driver = webdriver.Chrome(options=options)
driver.get("https://www.tendertiger.com/User/Account?login")
input("🔐 Please login and filter tenders by 'Odisha' on the dashboard. Then press ENTER to continue...")

wait = WebDriverWait(driver, 15)
summary = []

# === FIND ALL TENDERS ===
rows = driver.find_elements(By.CSS_SELECTOR, "#tblTenders > tr")

for i, row in enumerate(rows):
    try:
        tender_link = row.find_element(By.TAG_NAME, "a")
        onclick = tender_link.get_attribute("onclick")
        tid = onclick.split("'")[1]  # Extract TID like '20240530810'
        print(f"\n🔎 Processing TID: {tid}")

        # Open new tab and go to tender detail page
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])
        driver.get(f"https://www.tendertiger.com/TenderDetails.aspx?TID={tid}")
        time.sleep(3)

        # Make TID folder
        tdr_dir = os.path.join(base_download_dir, tid)
        os.makedirs(tdr_dir, exist_ok=True)

        # Gather file download links
        links = driver.find_elements(By.TAG_NAME, "a")
        boq_files = []
        other_files = []

        for link in links:
            href = link.get_attribute("href")
            if href and "DownloadFile" in href and "FileName=" in href:
                filename = href.split("FileName=")[-1]
                print(f"📥 Downloading: {filename}")

                try:
                    driver.get(href)
                    time.sleep(4)  # Let file download

                    # Move from base folder to TID folder
                    for f in os.listdir(base_download_dir):
                        if f.lower() in filename.lower():
                            src = os.path.join(base_download_dir, f)
                            dst = os.path.join(tdr_dir, f)
                            shutil.move(src, dst)
                            if "boq" in f.lower():
                                boq_files.append(f)
                            else:
                                other_files.append(f)
                except Exception as e:
                    print(f"⚠️ Failed downloading {filename}: {e}")

        summary.append({
            "TID": tid,
            "BOQ Files": ", ".join(boq_files) if boq_files else "Not Found",
            "Other Files": ", ".join(other_files) if other_files else "Not Found"
        })

        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(2)

    except Exception as e:
        print(f"❌ Error on row {i}: {e}")
        summary.append({
            "TID": f"Row {i}",
            "BOQ Files": "Error",
            "Other Files": str(e)
        })
        driver.switch_to.window(driver.window_handles[0])

# === SAVE SUMMARY ===
summary_path = os.path.join(base_download_dir, "TenderTiger_Summary.xlsx")
pd.DataFrame(summary).to_excel(summary_path, index=False)
print(f"\n✅ Done! Summary saved at: {summary_path}")

driver.quit()


🔐 Please login and filter tenders by 'Odisha' on the dashboard. Then press ENTER to continue... 



✅ Done! Summary saved at: C:\Users\Jaydeb\TenderTiger_Downloads\TenderTiger_Summary.xlsx


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, os, shutil
import pandas as pd

# === CONFIGURATION ===
base_download_dir = os.path.join(os.getcwd(), "TenderTiger_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "profile.default_content_settings.popups": 0,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)

# === START DRIVER ===
driver = webdriver.Chrome(options=options)
driver.get("https://www.tendertiger.com/User/Account?login")
input("🔐 Please login to TenderTiger manually. Then press ENTER to continue...")

wait = WebDriverWait(driver, 30)

# === SELECT ODISHA CHECKBOX USING LABEL ===
try:
    print("📜 Scrolling to top...")
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(2)

    print("⏳ Looking for label containing 'Odisha'...")
    odisha_label = wait.until(
        EC.presence_of_element_located((By.XPATH, "//label[contains(text(), 'Odisha')]"))
    )
    checkbox_id = odisha_label.get_attribute("for")
    odisha_checkbox = driver.find_element(By.ID, checkbox_id)

    if not odisha_checkbox.is_selected():
        driver.execute_script("arguments[0].click();", odisha_checkbox)
        print(f"✅ Selected checkbox: {checkbox_id}")

    # Click search
    try:
        search_button = driver.find_element(By.ID, "btnSearch")
        driver.execute_script("arguments[0].click();", search_button)
        print("🔄 Search triggered")
    except:
        print("⚠️ Search button not found, assuming auto-refresh")

except Exception as e:
    print("❌ Could not apply Odisha filter. Screenshot saved.")
    driver.save_screenshot("odisha_filter_error.png")
    driver.quit()
    raise e

# === WAIT FOR TENDERS TABLE ===
try:
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#tblTenders > tr")))
    print("✅ Tenders loaded")
except:
    print("❌ Tender table not found")
    driver.quit()
    exit()

summary = []
rows = driver.find_elements(By.CSS_SELECTOR, "#tblTenders > tr")
print(f"🔢 Total tenders found: {len(rows)}")

# === LOOP THROUGH TID ROWS ===
for i, row in enumerate(rows):
    try:
        tender_link = row.find_element(By.TAG_NAME, "a")
        onclick = tender_link.get_attribute("onclick")
        tid = onclick.split("'")[1]
        print(f"\n🔎 Processing TID: {tid}")

        # Open tender detail in new tab
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])
        driver.get(f"https://www.tendertiger.com/TenderDetails.aspx?TID={tid}")
        time.sleep(3)

        tdr_dir = os.path.join(base_download_dir, tid)
        os.makedirs(tdr_dir, exist_ok=True)

        # Collect document links
        links = driver.find_elements(By.TAG_NAME, "a")
        boq_files = []
        other_files = []

        for link in links:
            href = link.get_attribute("href")
            if href and "DownloadFile" in href and "FileName=" in href:
                filename = href.split("FileName=")[-1]
                print(f"📥 Downloading: {filename}")
                try:
                    driver.get(href)
                    time.sleep(4)
                    for f in os.listdir(base_download_dir):
                        if f.lower() in filename.lower():
                            shutil.move(
                                os.path.join(base_download_dir, f),
                                os.path.join(tdr_dir, f)
                            )
                            if "boq" in f.lower():
                                boq_files.append(f)
                            else:
                                other_files.append(f)
                except Exception as e:
                    print(f"⚠️ Failed downloading {filename}: {e}")

        summary.append({
            "TID": tid,
            "BOQ Files": ", ".join(boq_files) if boq_files else "Not Found",
            "Other Files": ", ".join(other_files) if other_files else "Not Found"
        })

        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(2)

    except Exception as e:
        print(f"❌ Error on row {i}: {e}")
        summary.append({
            "TID": f"Row {i}",
            "BOQ Files": "Error",
            "Other Files": str(e)
        })
        driver.switch_to.window(driver.window_handles[0])

# === SAVE SUMMARY TO EXCEL ===
summary_path = os.path.join(base_download_dir, "TenderTiger_Summary.xlsx")
pd.DataFrame(summary).to_excel(summary_path, index=False)
print(f"\n✅ Done! Summary saved at: {summary_path}")

driver.quit()


🔐 Please login to TenderTiger manually. Then press ENTER to continue... 


📜 Scrolling to top...
⏳ Looking for label containing 'Odisha'...
❌ Could not apply Odisha filter. Screenshot saved.


TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff7f8e0fe75+79173]
	GetHandleVerifier [0x0x7ff7f8e0fed0+79264]
	(No symbol) [0x0x7ff7f8bc9e5a]
	(No symbol) [0x0x7ff7f8c20586]
	(No symbol) [0x0x7ff7f8c2083c]
	(No symbol) [0x0x7ff7f8c74247]
	(No symbol) [0x0x7ff7f8c489af]
	(No symbol) [0x0x7ff7f8c7100d]
	(No symbol) [0x0x7ff7f8c48743]
	(No symbol) [0x0x7ff7f8c114c1]
	(No symbol) [0x0x7ff7f8c12253]
	GetHandleVerifier [0x0x7ff7f90da2ad+3004797]
	GetHandleVerifier [0x0x7ff7f90d46fd+2981325]
	GetHandleVerifier [0x0x7ff7f90f3350+3107360]
	GetHandleVerifier [0x0x7ff7f8e2a9fe+188622]
	GetHandleVerifier [0x0x7ff7f8e3228f+219487]
	GetHandleVerifier [0x0x7ff7f8e18dc4+115860]
	GetHandleVerifier [0x0x7ff7f8e18f79+116297]
	GetHandleVerifier [0x0x7ff7f8dff528+11256]
	BaseThreadInitThunk [0x0x7ffc413bdbe7+23]
	RtlUserThreadStart [0x0x7ffc420e5a4c+44]
